<a href="https://colab.research.google.com/github/Dra001ven/Team-6_Healthcare-hackathon/blob/main/eagles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
from google.colab import drive
print("All libraries imported successfully")

All libraries imported successfully


In [17]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
ct = pd.read_csv('/content/drive/MyDrive/contact_tracing.csv')
mob = pd.read_csv('/content/drive/MyDrive/mobility.csv')
vitals = pd.read_csv('/content/drive/MyDrive/vitals.csv')

print(f"contact_tracing: {ct.shape[0]:,} rows, {ct.shape[1]} columns")
print(f"mobility:        {mob.shape[0]:,} rows, {mob.shape[1]} columns")
print(f"vitals:          {vitals.shape[0]:,} rows, {vitals.shape[1]} columns")

contact_tracing: 119,338 rows, 9 columns
mobility:        50,737 rows, 11 columns
vitals:          2,307 rows, 12 columns


In [24]:
print("=== FIRST 5 ROWS ===")
print(ct.head())
print()
print("=== COLUMN INFO ===")
print(ct.info())
print()
print("=== MISSING VALUES ===")
print(ct.isnull().sum())
print()
print("=== PROXIMITY CATEGORIES ===")
print(ct['proximity'].value_counts())
print()
print(f"Unique users: {ct['user_id'].nunique()}")
print(f"Unique MACs: {ct['mac'].nunique()}")
print(f"Unscored RSSI (-1): {(ct['rssi'] == -1).sum()} ({(ct['rssi'] == -1).mean()*100:.1f}%)")

=== FIRST 5 ROWS ===
  user_id        date      time  latitude  longitude    geohash     mac  rssi  \
0    U002  2023-10-19  18:06:01      7.68       6.42  s1kedn1tq  D13941    -1   
1    U002  2023-10-19  18:06:01      7.68       6.42  s1kedn1tq  D13227     8   
2    U002  2023-10-19  18:06:01      7.68       6.42  s1kedn1tq  D21917    -1   
3    U002  2023-10-19  18:06:01      7.68       6.42  s1kedn1tq  D19180    -1   
4    U003  2023-10-20  06:12:13      5.37       7.00  s0uyx62pp  D05480     7   

  proximity  
0  unscored  
1     close  
2  unscored  
3  unscored  
4     close  

=== COLUMN INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119338 entries, 0 to 119337
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   user_id    119338 non-null  object 
 1   date       119338 non-null  object 
 2   time       119338 non-null  object 
 3   latitude   119338 non-null  float64
 4   longitude  119338 non-null

In [22]:
print("=== FIRST 5 ROWS ===")
print(vitals.head())
print()
print("=== TEMPERATURE STATUS ===")
print(vitals['temp_status'].value_counts())
print()
print("=== HEART RATE STATUS ===")
print(vitals['hr_status'].value_counts())
print()
print(f"Unique devices: {vitals['device_id'].nunique()}")

=== FIRST 5 ROWS ===
  device_id        date      time  latitude  longitude    geohash  \
0      D009  2024-05-13  06:56:50      7.30       5.14  s179sctvp   
1      D009  2024-05-13  06:57:00      7.30       5.14  s179sctus   
2      D009  2024-05-13  06:57:20      7.30       5.14  s179sctew   
3      D009  2024-05-13  07:16:00      7.30       5.13  s179scc4h   
4      D009  2024-05-13  07:17:25      7.30       5.13  s179scc1q   

   temperature temp_status  heartbeat hr_status  movement  battery  
0        31.00         low      96.00    normal      0.00   100.00  
1        31.00         low     110.00      high      0.00   100.00  
2        30.00         low      76.00    normal      1.00   100.00  
3        34.00         low      49.00       low      0.00   100.00  
4        34.00         low      43.00       low      0.00   100.00  

=== TEMPERATURE STATUS ===
temp_status
low       2002
normal     168
high       137
Name: count, dtype: int64

=== HEART RATE STATUS ===
hr_status
no

In [28]:
ct['timestamp'] = pd.to_datetime(ct['timestamp'])
ct['date'] = ct['timestamp'].dt.date
ct['hour'] = ct['timestamp'].dt.hour
ct['month'] = ct['timestamp'].dt.month
ct['rssi_clean'] = ct['rssi'].replace(-1, np.nan)

before = len(ct)
ct.drop_duplicates(inplace=True)
after = len(ct)

print(f"Duplicates removed: {before - after}")
print(f"Rows remaining: {after:,}")
print("Cleaning complete.")

Duplicates removed: 0
Rows remaining: 119,334
Cleaning complete.


In [30]:
mob['timestamp'] = pd.to_datetime(mob['date'].astype(str) + ' ' + mob['time'].astype(str))
mob['date'] = mob['timestamp'].dt.date
mob['hour'] = mob['timestamp'].dt.hour

before = len(mob)
mob = mob[
    (mob['latitude'].between(-90, 90)) &
    (mob['longitude'].between(-180, 180))
]
after = len(mob)

print(f"Rows removed for bad GPS: {before - after}")
print(f"Rows remaining: {after:,}")
print("Cleaning complete.")

Rows removed for bad GPS: 0
Rows remaining: 50,737
Cleaning complete.


In [32]:
vitals['timestamp'] = pd.to_datetime(vitals['date'].astype(str) + ' ' + vitals['time'].astype(str))
vitals['date'] = vitals['timestamp'].dt.date
vitals['hour'] = vitals['timestamp'].dt.hour

vitals['fever_flag'] = (
    (vitals['temp_status'] == 'high') &
    (vitals['hr_status'] == 'high')
).astype(int)

print(f"Total readings: {len(vitals):,}")
print(f"Fever flags: {vitals['fever_flag'].sum()}")
print(f"Fever flag rate: {vitals['fever_flag'].mean()*100:.1f}%")
print("Cleaning complete.")

Total readings: 2,307
Fever flags: 1
Fever flag rate: 0.0%
Cleaning complete.


In [33]:
ct_profile = ct.groupby('user_id').agg(
    total_detections_ct = ('mac', 'count'),
    unique_macs = ('mac', 'nunique'),
    close_contacts_ct = ('proximity',
                         lambda x: x.isin(['close', 'very close']).sum()),
    active_days_ct = ('date', 'nunique'),
    locations_visited = ('geohash', 'nunique'),
    avg_rssi = ('rssi_clean', 'mean')
).reset_index()

ct_profile['close_contact_rate'] = (
    ct_profile['close_contacts_ct'] / ct_profile['total_detections_ct']
)
ct_profile['detections_per_day'] = (
    ct_profile['total_detections_ct'] / ct_profile['active_days_ct']
)

print(f"User profiles built: {len(ct_profile)}")
print()
print(ct_profile.head())

User profiles built: 88

  user_id  total_detections_ct  unique_macs  close_contacts_ct  \
0    U001                  173           75                 23   
1    U002                    4            4                  1   
2    U003                  548           40                241   
3    U004                    1            1                  1   
4    U005                   33           15                 12   

   active_days_ct  locations_visited  avg_rssi  close_contact_rate  \
0               3                  8      6.58                0.13   
1               1                  1      8.00                0.25   
2               1                 75     10.21                0.44   
3               1                  1      8.00                1.00   
4               1                  9     11.00                0.36   

   detections_per_day  
0               57.67  
1                4.00  
2              548.00  
3                1.00  
4               33.00  


In [38]:
mob_profile = mob.groupby('user_id').agg(
    total_mobility_records = ('latitude', 'count'),
    unique_locations_mob = ('geohash', 'nunique'),
    avg_daily_exposure = ('daily_exposure', 'mean'),
    total_active_time_mins = ('daily_exposure', 'sum'), # Using daily_exposure as a proxy for active time
    active_days_mob = ('date', 'nunique')
).reset_index()

combined = pd.merge(ct_profile, mob_profile, on='user_id', how='outer')
combined.fillna(0, inplace=True);

print(f"Combined user profiles: {len(combined)}")
print()
print(combined.head())

Combined user profiles: 88

  user_id  total_detections_ct  unique_macs  close_contacts_ct  \
0    U001                  173           75                 23   
1    U002                    4            4                  1   
2    U003                  548           40                241   
3    U004                    1            1                  1   
4    U005                   33           15                 12   

   active_days_ct  locations_visited  avg_rssi  close_contact_rate  \
0               3                  8      6.58                0.13   
1               1                  1      8.00                0.25   
2               1                 75     10.21                0.44   
3               1                  1      8.00                1.00   
4               1                  9     11.00                0.36   

   detections_per_day  total_mobility_records  unique_locations_mob  \
0               57.67                   49.00                 37.00   
1           

In [36]:
feature_docs = pd.DataFrame({
    'Feature': [
        'total_detections_ct',
        'unique_macs',
        'close_contacts_ct',
        'close_contact_rate',
        'active_days_ct',
        'detections_per_day',
        'locations_visited',
        'avg_rssi',
        'exposure_score',
        'daily_exposure',
        'contact_ping_rate'
    ],
    'Source': [
        'contact_tracing.csv',
        'contact_tracing.csv',
        'contact_tracing.csv',
        'contact_tracing.csv',
        'contact_tracing.csv',
        'contact_tracing.csv',
        'contact_tracing.csv',
        'contact_tracing.csv',
        'mobility.csv',
        'mobility.csv',
        'mobility.csv'
    ],
    'Formula': [
        'Count of all rows per user_id',
        'Count of unique mac values per user_id',
        'Count where proximity is close or very close',
        'close_contacts_ct divided by total_detections_ct',
        'Count of unique dates per user_id',
        'total_detections_ct divided by active_days_ct',
        'Count of unique geohash values per user_id',
        'Mean of rssi_clean (excluding -1 values)',
        'Pre-computed in mobility.csv — extracted as-is',
        'Pre-computed in mobility.csv — extracted as-is',
        'Sum of has_contact divided by count of all pings'
    ],
    'Purpose': [
        'Measures how active each user was overall',
        'Measures how many different people or devices were encountered',
        'Counts only genuinely high-risk close-range contacts',
        'Proportion of contacts that were high risk',
        'How many days the user was actively using the device',
        'Normalises detection count across users active for different durations',
        'How many different locations the user moved through',
        'Average signal strength — proxy for how close contacts were',
        'Total weighted exposure risk across entire study period',
        'Daily normalised exposure — most important risk indicator',
        'Proportion of location pings that coincided with contact events'
    ]
})

print("=== FEATURE ENGINEERING DOCUMENTATION (Required Deliverable) ===")
print()
feature_docs

=== FEATURE ENGINEERING DOCUMENTATION (Required Deliverable) ===



,Feature,Source,Formula,Purpose
0,total_detections_ct,contact_tracing.csv,Count of all rows per user_id,Measures how active each user was overall
1,unique_macs,contact_tracing.csv,Count of unique mac values per user_id,Measures how many different people or devices ...
2,close_contacts_ct,contact_tracing.csv,Count where proximity is close or very close,Counts only genuinely high-risk close-range co...
3,close_contact_rate,contact_tracing.csv,close_contacts_ct divided by total_detections_ct,Proportion of contacts that were high risk
4,active_days_ct,contact_tracing.csv,Count of unique dates per user_id,How many days the user was actively using the ...
5,detections_per_day,contact_tracing.csv,total_detections_ct divided by active_days_ct,Normalises detection count across users active...
6,locations_visited,contact_tracing.csv,Count of unique geohash values per user_id,How many different locations the user moved th...
7,avg_rssi,contact_tracing.csv,Mean of rssi_clean (excluding -1 values),Average signal strength — proxy for how close ...
8,exposure_score,mobility.csv,Pre-computed in mobility.csv — extracted as-is,Total weighted exposure risk across entire stu...
9,daily_exposure,mobility.csv,Pre-computed in mobility.csv — extracted as-is,Daily normalised exposure — most important ris...
